# 🥗 AI Meal Planner — EDA Pipeline
**COMP6056001 — Artificial Intelligence | Group A**

This notebook performs a full Exploratory Data Analysis across all datasets used in the project:
- `FOOD-DATA-GROUP 1–5` — ingredient nutrition lookup
- `daily_food_nutrition_dataset.csv` — meal-level nutrition with meal type labels
- `combined_laboratory.csv` — NHANES lab results (glucose, cholesterol, HbA1c, blood markers)
- `NHANES 2021–2023` — downloaded directly (BMI, disease flags, dietary intake)

All plots are saved to `EDA_Results/` which is synced to your Google Drive AI folder.

## 0. Setup — Mount Drive & Install Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── CONFIGURE YOUR PATHS HERE ──────────────────────────────────────────────
# Path to your AI folder in Google Drive (update if different)
AI_FOLDER   = '/content/drive/MyDrive/AI'
# Where datasets are (could be MyDrive or Shared with me)
DATA_FOLDER = '/content/drive/MyDrive/AI'         # update if in a subfolder
SHARED_DATA = '/content/drive/Shareddrives'       # fallback for shared drives
# Output folder (will be created inside AI_FOLDER)
OUT_FOLDER  = os.path.join(AI_FOLDER, 'EDA_Results')
# ───────────────────────────────────────────────────────────────────────────

os.makedirs(OUT_FOLDER, exist_ok=True)
print(f'Output folder ready: {OUT_FOLDER}')

In [ ]:
!pip install -q missingno seaborn scipy

In [ ]:
import pandas as pd
import numpy  as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import glob
import warnings
import zipfile
import io
from scipy import stats

warnings.filterwarnings('ignore')
matplotlib.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.titlesize':     13,
    'axes.labelsize':     11,
    'xtick.labelsize':    10,
    'ytick.labelsize':    10,
})

# Colour palette consistent with project
PURPLE = '#534AB7'
TEAL   = '#1D9E75'
CORAL  = '#D85A30'
AMBER  = '#EF9F27'
PAL    = [PURPLE, TEAL, CORAL, AMBER, '#378ADD', '#639922']
sns.set_palette(PAL)

def save(name):
    """Save current figure to EDA_Results folder."""
    path = os.path.join(OUT_FOLDER, f'{name}.png')
    plt.savefig(path, bbox_inches='tight', dpi=150)
    print(f'  Saved → {path}')
    plt.show()

print('Libraries loaded ✓')

---
## 1. Load All Datasets

### 1a. FOOD-DATA-GROUP 1–5 (unzip from Drive)

In [ ]:
# ── Option A: files already extracted (CSV files in DATA_FOLDER) ──
group_files = sorted(glob.glob(os.path.join(DATA_FOLDER, 'FOOD-DATA-GROUP*.csv')))

# ── Option B: still inside the zip ──
if not group_files:
    zip_path = glob.glob(os.path.join(DATA_FOLDER, '*foundation_food*.zip'))
    if zip_path:
        extract_dir = '/content/food_data_extracted'
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path[0]) as z:
            z.extractall(extract_dir)
        group_files = sorted(glob.glob(os.path.join(extract_dir, 'FOOD-DATA-GROUP*.csv')))
        print(f'Extracted {len(group_files)} GROUP files from zip')

if not group_files:
    raise FileNotFoundError('FOOD-DATA-GROUP CSVs not found. Check DATA_FOLDER path above.')

food_raw = pd.concat(
    [pd.read_csv(f) for f in group_files],
    ignore_index=True
)
# Drop duplicate unnamed index columns
food_raw = food_raw.loc[:, ~food_raw.columns.str.contains('^Unnamed')]

print(f'FOOD-DATA merged:  {food_raw.shape[0]:,} rows × {food_raw.shape[1]} cols')
food_raw.head(3)

### 1b. Daily Food Nutrition Dataset

In [ ]:
meal_path = glob.glob(os.path.join(DATA_FOLDER, 'daily_food_nutrition_dataset.csv'))
if not meal_path:
    # Also try extracted dir
    meal_path = glob.glob('/content/food_data_extracted/daily_food_nutrition_dataset.csv')

if not meal_path:
    raise FileNotFoundError('daily_food_nutrition_dataset.csv not found.')

meal_df = pd.read_csv(meal_path[0], on_bad_lines='skip')
print(f'Meal dataset:  {meal_df.shape[0]:,} rows × {meal_df.shape[1]} cols')
meal_df.head(3)

### 1c. NHANES Combined Laboratory Data

In [ ]:
lab_path = glob.glob(os.path.join(DATA_FOLDER, 'combined_laboratory.csv'))
if not lab_path:
    lab_path = glob.glob('/content/food_data_extracted/combined_laboratory.csv')

if not lab_path:
    raise FileNotFoundError('combined_laboratory.csv not found.')

lab_df = pd.read_csv(lab_path[0], on_bad_lines='skip')
print(f'Laboratory data: {lab_df.shape[0]:,} rows × {lab_df.shape[1]} cols')

# Key columns we care about
LAB_COLS = {
    'SEQN':    'Participant ID',
    'LBXGH':   'HbA1c % (diabetes marker)',
    'LBXGLU':  'Fasting glucose mg/dL',
    'LBXTC':   'Total cholesterol mg/dL',
    'LBDHDD':  'HDL cholesterol mg/dL',
    'LBXHGB':  'Hemoglobin g/dL',
    'LBXHSCRP':'High-sensitivity CRP mg/L (inflammation)',
    'LBXVIDMS':'Vitamin D nmol/L',
    'LBXFER':  'Ferritin ng/mL (iron stores)',
    'PHAFSTHR':'Fasting time hours',
}
available_lab = {k: v for k, v in LAB_COLS.items() if k in lab_df.columns}
print(f'Key lab columns available: {list(available_lab.keys())}')
lab_df[list(available_lab.keys())].head(3)

### 1d. NHANES 2021–2023 (Download directly from CDC)

In [ ]:
BASE = 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/'

print('Downloading NHANES 2021–2023 files from CDC...')
nhanes_demo = pd.read_sas(BASE + 'DEMO_L.XPT')
print(f'  Demographics:   {nhanes_demo.shape}')

nhanes_bmi  = pd.read_sas(BASE + 'BMX_L.XPT')
print(f'  Body measures:  {nhanes_bmi.shape}')

nhanes_diab = pd.read_sas(BASE + 'DIQ_L.XPT')
print(f'  Diabetes:       {nhanes_diab.shape}')

nhanes_bp   = pd.read_sas(BASE + 'BPQ_L.XPT')
print(f'  Blood pressure: {nhanes_bp.shape}')

nhanes_diet = pd.read_sas(BASE + 'DR1TOT_L.XPT')
print(f'  Dietary intake: {nhanes_diet.shape}')

# Merge all on participant ID
nhanes = (
    nhanes_demo
    .merge(nhanes_bmi[['SEQN','BMXBMI','BMXWT','BMXHT']], on='SEQN', how='left')
    .merge(nhanes_diab[['SEQN','DIQ010','DIQ160','DIQ170']], on='SEQN', how='left')
    .merge(nhanes_bp[['SEQN','BPQ020','BPQ030']], on='SEQN', how='left')
    .merge(nhanes_diet[['SEQN','DR1TKCAL','DR1TPROT','DR1TCARB','DR1TTFAT','DR1TFIBE',
                         'DR1TSODI','DR1TPOTA','DR1TCALC','DR1TIRON']], on='SEQN', how='left')
)

# Also merge with lab data if SEQN available
if 'SEQN' in lab_df.columns:
    lab_key_cols = ['SEQN'] + [c for c in ['LBXGH','LBXGLU','LBXTC','LBDHDD','LBXHGB','LBXHSCRP'] if c in lab_df.columns]
    nhanes = nhanes.merge(lab_df[lab_key_cols], on='SEQN', how='left')
    print(f'Lab data merged on SEQN ✓')

print(f'\nFull NHANES merged: {nhanes.shape[0]:,} rows × {nhanes.shape[1]} cols')
nhanes.head(3)

---
## 2. Dataset Summaries

In [ ]:
def summary_table(df, name):
    print(f'\n{'='*55}')
    print(f'  {name}')
    print(f'{'='*55}')
    print(f'  Shape:           {df.shape[0]:,} rows × {df.shape[1]} columns')
    print(f'  Memory:          {df.memory_usage(deep=True).sum()/1e6:.1f} MB')
    total_missing = df.isnull().sum().sum()
    total_cells   = df.shape[0] * df.shape[1]
    print(f'  Missing values:  {total_missing:,} / {total_cells:,} ({100*total_missing/total_cells:.1f}%)')
    dup = df.duplicated().sum()
    print(f'  Duplicate rows:  {dup:,}')
    num_cols = df.select_dtypes('number').shape[1]
    cat_cols = df.select_dtypes('object').shape[1]
    print(f'  Numeric cols:    {num_cols}   |   Categorical cols: {cat_cols}')

summary_table(food_raw,  'FOOD-DATA-GROUP 1–5 (merged)')
summary_table(meal_df,   'Daily Food Nutrition Dataset')
summary_table(lab_df,    'NHANES Combined Laboratory')
summary_table(nhanes,    'NHANES 2021–2023 (merged)')

---
## 3. Missing Value Analysis

In [ ]:
# ── 3a. NHANES missing values heatmap ──
nhanes_key_cols = [
    'BMXBMI','BMXWT','BMXHT',
    'DIQ010','BPQ020',
    'DR1TKCAL','DR1TPROT','DR1TCARB','DR1TTFAT','DR1TFIBE',
    'DR1TSODI','DR1TPOTA','DR1TCALC','DR1TIRON',
    'RIDAGEYR','RIAGENDR','RIDRETH3',
]
nhanes_key_cols = [c for c in nhanes_key_cols if c in nhanes.columns]

miss = nhanes[nhanes_key_cols].isnull().mean().sort_values(ascending=True) * 100

fig, ax = plt.subplots(figsize=(10, 5))
colors = [CORAL if v > 30 else AMBER if v > 10 else TEAL for v in miss.values]
bars = ax.barh(miss.index, miss.values, color=colors, height=0.6)
ax.set_xlabel('Missing values (%)')
ax.set_title('NHANES 2021–23 — Missing values by key column')
ax.axvline(10, color=AMBER, linestyle='--', linewidth=0.8, alpha=0.7, label='>10% threshold')
ax.axvline(30, color=CORAL, linestyle='--', linewidth=0.8, alpha=0.7, label='>30% threshold')
for bar, val in zip(bars, miss.values):
    if val > 0:
        ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9)
ax.legend(fontsize=9)
plt.tight_layout()
save('01_nhanes_missing_values')

In [ ]:
# ── 3b. Food data missing values ──
food_miss = food_raw.isnull().mean().sort_values(ascending=False) * 100
food_miss = food_miss[food_miss > 0]

if len(food_miss) > 0:
    fig, ax = plt.subplots(figsize=(10, max(3, len(food_miss)*0.4)))
    ax.barh(food_miss.index, food_miss.values, color=AMBER, height=0.6)
    ax.set_xlabel('Missing values (%)')
    ax.set_title('FOOD-DATA-GROUP — Missing values')
    plt.tight_layout()
    save('02_food_missing_values')
else:
    print('✓ FOOD-DATA-GROUP: No missing values found.')

In [ ]:
# ── 3c. Missingness matrix (visual pattern) ──
fig, ax = plt.subplots(figsize=(12, 5))
msno.matrix(nhanes[nhanes_key_cols].sample(min(500, len(nhanes))),
            ax=ax, color=(0.32, 0.29, 0.72), fontsize=9)
ax.set_title('NHANES — missingness pattern (500 sample rows)')
save('03_nhanes_missing_matrix')

---
## 4. BMI Distribution Analysis

In [ ]:
bmi = nhanes['BMXBMI'].dropna()
bmi = bmi[(bmi > 10) & (bmi < 70)]  # remove physiologically impossible values

def bmi_cat(v):
    if v < 18.5: return 'Underweight'
    elif v < 25: return 'Normal'
    elif v < 30: return 'Overweight'
    else:        return 'Obese'

bmi_series = nhanes.loc[nhanes['BMXBMI'].between(10,70), 'BMXBMI']
nhanes_clean = nhanes.loc[nhanes['BMXBMI'].between(10,70)].copy()
nhanes_clean['bmi_category'] = nhanes_clean['BMXBMI'].apply(bmi_cat)

cat_order  = ['Underweight', 'Normal', 'Overweight', 'Obese']
cat_colors = [TEAL, '#3B6D11', AMBER, CORAL]
cat_counts = nhanes_clean['bmi_category'].value_counts()[cat_order]
cat_pct    = (cat_counts / cat_counts.sum() * 100).round(1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Histogram with KDE
axes[0].hist(bmi_series, bins=50, color=PURPLE, alpha=0.75, edgecolor='white', linewidth=0.4)
ax2 = axes[0].twinx()
bmi_series.plot.kde(ax=ax2, color=TEAL, linewidth=2)
ax2.set_ylabel('Density', color=TEAL)
ax2.tick_params(axis='y', colors=TEAL)
axes[0].axvline(18.5, color=TEAL,   linestyle='--', linewidth=1, alpha=0.8)
axes[0].axvline(25,   color=AMBER,  linestyle='--', linewidth=1, alpha=0.8)
axes[0].axvline(30,   color=CORAL,  linestyle='--', linewidth=1, alpha=0.8)
axes[0].set_xlabel('BMI (kg/m²)')
axes[0].set_ylabel('Count')
axes[0].set_title('BMI distribution')

# Bar chart by category
bars = axes[1].bar(cat_order, cat_counts.values, color=cat_colors, width=0.55)
for bar, pct in zip(bars, cat_pct.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
                 f'{pct}%', ha='center', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].set_title('BMI category breakdown')

# Box plot
data_by_cat = [nhanes_clean.loc[nhanes_clean['bmi_category']==c,'BMXBMI'].dropna() for c in cat_order]
bp = axes[2].boxplot(data_by_cat, patch_artist=True, labels=cat_order,
                      medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], cat_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[2].set_ylabel('BMI')
axes[2].set_title('BMI spread by category')

fig.suptitle('BMI Analysis — NHANES 2021–2023', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save('04_bmi_distribution')

print(f'\nBMI statistics:')
print(bmi_series.describe().round(2).to_string())
print(f'\nSkewness: {bmi_series.skew():.3f}  |  Kurtosis: {bmi_series.kurt():.3f}')

---
## 5. Disease Prevalence Analysis

In [ ]:
# Recode disease flags
# DIQ010: 1=Yes diabetes, 2=No, 3=Borderline
# BPQ020: 1=Yes hypertension, 2=No
nhanes_clean['has_diabetes']     = (nhanes_clean['DIQ010'] == 1).astype(int) if 'DIQ010' in nhanes_clean.columns else np.nan
nhanes_clean['has_borderline']   = (nhanes_clean['DIQ010'] == 3).astype(int) if 'DIQ010' in nhanes_clean.columns else np.nan
nhanes_clean['has_hypertension'] = (nhanes_clean['BPQ020'] == 1).astype(int) if 'BPQ020' in nhanes_clean.columns else np.nan
nhanes_clean['has_malnutrition'] = ((nhanes_clean['BMXBMI'] < 18.5)).astype(int)

disease_cols = ['has_diabetes', 'has_borderline', 'has_hypertension', 'has_malnutrition']
disease_cols = [c for c in disease_cols if c in nhanes_clean.columns and nhanes_clean[c].notna().any()]

disease_rates = {c: nhanes_clean[c].mean()*100 for c in disease_cols}
labels = ['Diabetes', 'Borderline\nDiabetes', 'Hypertension', 'Malnutrition']
labels = labels[:len(disease_cols)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Prevalence bar chart
colors_d = [CORAL, AMBER, PURPLE, TEAL]
bars = axes[0].bar(labels, list(disease_rates.values()), color=colors_d[:len(disease_cols)], width=0.55)
for bar, val in zip(bars, disease_rates.values()):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Prevalence (%)')
axes[0].set_title('Disease prevalence in NHANES sample')

# Co-morbidity: diabetes + hypertension
if 'has_diabetes' in nhanes_clean.columns and 'has_hypertension' in nhanes_clean.columns:
    comorbid_subset = nhanes_clean[['has_diabetes','has_hypertension']].dropna()
    groups = [
        ('Neither',              (~comorbid_subset['has_diabetes'].astype(bool)) & (~comorbid_subset['has_hypertension'].astype(bool))),
        ('Diabetes only',         comorbid_subset['has_diabetes'].astype(bool)  & (~comorbid_subset['has_hypertension'].astype(bool))),
        ('Hypertension only',    (~comorbid_subset['has_diabetes'].astype(bool)) &  comorbid_subset['has_hypertension'].astype(bool)),
        ('Both',                  comorbid_subset['has_diabetes'].astype(bool)  &   comorbid_subset['has_hypertension'].astype(bool)),
    ]
    group_counts = [mask.sum() for _, mask in groups]
    group_labels = [g[0] for g in groups]
    wedge_colors = [TEAL, CORAL, PURPLE, AMBER]
    axes[1].pie(group_counts, labels=group_labels, colors=wedge_colors,
                autopct='%1.1f%%', startangle=140,
                textprops={'fontsize': 10},
                wedgeprops={'linewidth': 1, 'edgecolor': 'white'})
    axes[1].set_title('Diabetes × Hypertension co-morbidity')

plt.suptitle('Disease Prevalence — NHANES 2021–2023', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
save('05_disease_prevalence')

---
## 6. BMI vs Disease Relationship

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# BMI by diabetes status
if 'has_diabetes' in nhanes_clean.columns:
    bmi_no_diab  = nhanes_clean.loc[nhanes_clean['has_diabetes']==0, 'BMXBMI'].dropna()
    bmi_yes_diab = nhanes_clean.loc[nhanes_clean['has_diabetes']==1, 'BMXBMI'].dropna()

    axes[0].hist(bmi_no_diab,  bins=40, alpha=0.65, color=TEAL,   label=f'No diabetes (n={len(bmi_no_diab):,})',  density=True)
    axes[0].hist(bmi_yes_diab, bins=40, alpha=0.65, color=CORAL,  label=f'Has diabetes (n={len(bmi_yes_diab):,})', density=True)
    axes[0].axvline(bmi_no_diab.median(),  color=TEAL,  linestyle='--', linewidth=1.5, alpha=0.9)
    axes[0].axvline(bmi_yes_diab.median(), color=CORAL, linestyle='--', linewidth=1.5, alpha=0.9)
    axes[0].set_xlabel('BMI (kg/m²)')
    axes[0].set_ylabel('Density')
    axes[0].set_title('BMI distribution by diabetes status')
    axes[0].legend()

    # Mann-Whitney U test
    stat, p = stats.mannwhitneyu(bmi_no_diab, bmi_yes_diab, alternative='two-sided')
    axes[0].text(0.98, 0.97, f'p = {p:.2e}', transform=axes[0].transAxes,
                 ha='right', va='top', fontsize=9,
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

# BMI by hypertension status
if 'has_hypertension' in nhanes_clean.columns:
    bmi_no_bp  = nhanes_clean.loc[nhanes_clean['has_hypertension']==0, 'BMXBMI'].dropna()
    bmi_yes_bp = nhanes_clean.loc[nhanes_clean['has_hypertension']==1, 'BMXBMI'].dropna()

    axes[1].hist(bmi_no_bp,  bins=40, alpha=0.65, color=TEAL,   label=f'No hypertension (n={len(bmi_no_bp):,})',  density=True)
    axes[1].hist(bmi_yes_bp, bins=40, alpha=0.65, color=PURPLE, label=f'Has hypertension (n={len(bmi_yes_bp):,})', density=True)
    axes[1].axvline(bmi_no_bp.median(),  color=TEAL,   linestyle='--', linewidth=1.5, alpha=0.9)
    axes[1].axvline(bmi_yes_bp.median(), color=PURPLE, linestyle='--', linewidth=1.5, alpha=0.9)
    axes[1].set_xlabel('BMI (kg/m²)')
    axes[1].set_ylabel('Density')
    axes[1].set_title('BMI distribution by hypertension status')
    axes[1].legend()

    stat, p = stats.mannwhitneyu(bmi_no_bp, bmi_yes_bp, alternative='two-sided')
    axes[1].text(0.98, 0.97, f'p = {p:.2e}', transform=axes[1].transAxes,
                 ha='right', va='top', fontsize=9,
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.suptitle('BMI vs Disease Status', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
save('06_bmi_vs_disease')

---
## 7. Dietary Intake (Macro) Analysis

In [ ]:
MACRO_COLS = {
    'DR1TKCAL': 'Calories (kcal)',
    'DR1TPROT': 'Protein (g)',
    'DR1TCARB': 'Carbohydrates (g)',
    'DR1TTFAT': 'Fat (g)',
    'DR1TFIBE': 'Fiber (g)',
    'DR1TSODI': 'Sodium (mg)',
}
available_macros = {k: v for k, v in MACRO_COLS.items() if k in nhanes_clean.columns}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, (col, label) in enumerate(available_macros.items()):
    data = nhanes_clean[col].dropna()
    # Cap at 99th percentile to remove extreme outliers for plotting
    cap = data.quantile(0.99)
    data_capped = data[data <= cap]

    axes[i].hist(data_capped, bins=45, color=PAL[i % len(PAL)], alpha=0.8, edgecolor='white', linewidth=0.3)
    axes[i].axvline(data.median(), color='black', linestyle='--', linewidth=1.2, alpha=0.7, label=f'Median: {data.median():.0f}')
    axes[i].axvline(data.mean(),   color='gray',  linestyle=':',  linewidth=1.2, alpha=0.7, label=f'Mean: {data.mean():.0f}')
    axes[i].set_title(label)
    axes[i].set_xlabel(label)
    axes[i].set_ylabel('Count')
    axes[i].legend(fontsize=8)

# Hide any empty subplots
for j in range(len(available_macros), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Daily Dietary Intake Distributions — NHANES 2021–2023', fontsize=14, fontweight='bold')
plt.tight_layout()
save('07_macro_distributions')

In [ ]:
# ── Macros by BMI category ──
bmi_cat_order = ['Underweight', 'Normal', 'Overweight', 'Obese']
fig, axes = plt.subplots(1, min(4, len(available_macros)), figsize=(14, 5))
if len(available_macros) == 1:
    axes = [axes]

plot_macros = list(available_macros.items())[:4]
for i, (col, label) in enumerate(plot_macros):
    plot_data = nhanes_clean[nhanes_clean['bmi_category'].isin(bmi_cat_order)]
    group_data = [plot_data.loc[plot_data['bmi_category']==cat, col].dropna() for cat in bmi_cat_order]
    group_data = [g for g in group_data if len(g) > 0]
    active_cats = [c for c, g in zip(bmi_cat_order, [plot_data.loc[plot_data['bmi_category']==cat, col].dropna() for cat in bmi_cat_order]) if len(g) > 0]

    bp = axes[i].boxplot(group_data, patch_artist=True, labels=active_cats,
                          medianprops=dict(color='white', linewidth=2),
                          flierprops=dict(marker='.', markersize=2, alpha=0.3))
    bmc = [TEAL, '#3B6D11', AMBER, CORAL]
    for patch, color in zip(bp['boxes'], bmc[:len(active_cats)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.85)
    axes[i].set_title(label, fontsize=11)
    axes[i].set_xlabel('BMI category')
    axes[i].tick_params(axis='x', rotation=15)

plt.suptitle('Daily Macros by BMI Category', fontsize=14, fontweight='bold')
plt.tight_layout()
save('08_macros_by_bmi_category')

---
## 8. Correlation Analysis

In [ ]:
corr_cols = [
    'BMXBMI',
    'DR1TKCAL','DR1TPROT','DR1TCARB','DR1TTFAT','DR1TFIBE','DR1TSODI',
    'has_diabetes','has_hypertension',
]
# Add lab cols if available
for c in ['LBXGH','LBXGLU','LBXTC','LBDHDD','LBXHGB']:
    if c in nhanes_clean.columns:
        corr_cols.append(c)

corr_cols  = [c for c in corr_cols if c in nhanes_clean.columns]
corr_labels = {
    'BMXBMI':'BMI','DR1TKCAL':'Calories','DR1TPROT':'Protein',
    'DR1TCARB':'Carbs','DR1TTFAT':'Fat','DR1TFIBE':'Fiber',
    'DR1TSODI':'Sodium','has_diabetes':'Diabetes','has_hypertension':'Hypertension',
    'LBXGH':'HbA1c','LBXGLU':'Glucose','LBXTC':'Cholesterol',
    'LBDHDD':'HDL','LBXHGB':'Hemoglobin'
}

corr_df = nhanes_clean[corr_cols].rename(columns=corr_labels).dropna(how='all')
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
cmap = sns.diverging_palette(250, 15, s=75, l=40, n=9, center='light', as_cmap=True)
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap=cmap, vmin=-1, vmax=1, center=0,
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 8}, ax=ax
)
ax.set_title('Correlation heatmap — BMI, macros, disease markers', fontsize=13, pad=12)
plt.tight_layout()
save('09_correlation_heatmap')

print('\nTop correlations with BMI:')
bmi_corr = corr_matrix['BMI'].drop('BMI').sort_values(key=abs, ascending=False)
print(bmi_corr.round(3).to_string())

---
## 9. Food Database (FOOD-DATA-GROUP) EDA

In [ ]:
# ── 9a. Nutritional density distribution ──
food_num = food_raw.select_dtypes('number')
key_food_cols = ['Caloric Value','Protein','Fat','Carbohydrates','Dietary Fiber',
                  'Sodium','Sugars','Nutrition Density']
key_food_cols = [c for c in key_food_cols if c in food_raw.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(key_food_cols):
    data = food_raw[col].dropna()
    cap  = data.quantile(0.98)
    data_c = data[data <= cap]
    axes[i].hist(data_c, bins=40, color=PAL[i % len(PAL)], alpha=0.8, edgecolor='white', linewidth=0.3)
    axes[i].axvline(data.median(), color='black', linestyle='--', linewidth=1, alpha=0.7)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('per 100g')

for j in range(len(key_food_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('FOOD-DATA-GROUP — Nutrient distributions (per 100g)', fontsize=14, fontweight='bold')
plt.tight_layout()
save('10_food_nutrient_distributions')

In [ ]:
# ── 9b. Top 20 highest & lowest calorie foods ──
if 'food' in food_raw.columns and 'Caloric Value' in food_raw.columns:
    food_sorted = food_raw[['food','Caloric Value','Protein','Fat','Dietary Fiber']].dropna(subset=['Caloric Value'])
    top20  = food_sorted.nlargest(20, 'Caloric Value')
    bot20  = food_sorted.nsmallest(20, 'Caloric Value')

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    axes[0].barh(top20['food'].str[:35], top20['Caloric Value'], color=CORAL, alpha=0.85)
    axes[0].set_xlabel('Caloric Value (per 100g)')
    axes[0].set_title('Top 20 highest calorie foods')
    axes[0].invert_yaxis()

    axes[1].barh(bot20['food'].str[:35], bot20['Caloric Value'], color=TEAL, alpha=0.85)
    axes[1].set_xlabel('Caloric Value (per 100g)')
    axes[1].set_title('Top 20 lowest calorie foods')
    axes[1].invert_yaxis()

    plt.suptitle('Caloric Extremes in Food Database', fontsize=14, fontweight='bold')
    plt.tight_layout()
    save('11_food_caloric_extremes')

In [ ]:
# ── 9c. Protein vs Calories scatter (colored by fiber) ──
if all(c in food_raw.columns for c in ['Caloric Value','Protein','Dietary Fiber']):
    sample = food_raw[['food','Caloric Value','Protein','Dietary Fiber']].dropna()
    sample = sample[sample['Caloric Value'] < 700]  # remove outliers

    fig, ax = plt.subplots(figsize=(10, 7))
    sc = ax.scatter(sample['Caloric Value'], sample['Protein'],
                    c=sample['Dietary Fiber'], cmap='YlGn',
                    alpha=0.6, s=18, edgecolors='none')
    plt.colorbar(sc, ax=ax, label='Dietary Fiber (g per 100g)')
    ax.set_xlabel('Caloric Value (per 100g)')
    ax.set_ylabel('Protein (g per 100g)')
    ax.set_title('Protein vs Calories — coloured by Fiber content')
    plt.tight_layout()
    save('12_food_protein_vs_calories')

---
## 10. Meal Type Analysis (Daily Food Nutrition Dataset)

In [ ]:
meal_num_cols = ['Calories (kcal)','Protein (g)','Carbohydrates (g)','Fat (g)','Fiber (g)','Sodium (mg)']
meal_num_cols = [c for c in meal_num_cols if c in meal_df.columns]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Count by meal type
meal_counts = meal_df['Meal_Type'].value_counts()
axes[0].bar(meal_counts.index, meal_counts.values,
            color=[PURPLE, TEAL, CORAL, AMBER][:len(meal_counts)], width=0.55)
for i, (idx, val) in enumerate(meal_counts.items()):
    axes[0].text(i, val + 3, str(val), ha='center', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_title('Meals by type')

# Calories by meal type
if 'Calories (kcal)' in meal_df.columns:
    meal_order = meal_df['Meal_Type'].unique().tolist()
    cal_by_type = [meal_df.loc[meal_df['Meal_Type']==m, 'Calories (kcal)'].dropna() for m in meal_order]
    bp = axes[1].boxplot(cal_by_type, patch_artist=True, labels=meal_order,
                          medianprops=dict(color='white', linewidth=2))
    for patch, color in zip(bp['boxes'], [PURPLE, TEAL, CORAL, AMBER]):
        patch.set_facecolor(color)
        patch.set_alpha(0.85)
    axes[1].set_ylabel('Calories (kcal)')
    axes[1].set_title('Calorie distribution by meal type')

# Macro radar-style bar chart (means per meal type)
if 'Meal_Type' in meal_df.columns:
    macro_means = meal_df.groupby('Meal_Type')[['Protein (g)','Carbohydrates (g)','Fat (g)']].mean()
    macro_means.plot(kind='bar', ax=axes[2], color=[CORAL, AMBER, PURPLE], width=0.6, edgecolor='white')
    axes[2].set_title('Average macros by meal type')
    axes[2].set_ylabel('Grams')
    axes[2].tick_params(axis='x', rotation=15)
    axes[2].legend(fontsize=9)

plt.suptitle('Meal Type Analysis — Daily Food Nutrition Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
save('13_meal_type_analysis')

---
## 11. Lab Markers Analysis (NHANES)

In [ ]:
lab_plot_cols = {
    'LBXGH':   ('HbA1c (%)', 'Diabetes threshold: 6.5%', 6.5),
    'LBXGLU':  ('Fasting Glucose (mg/dL)', 'Diabetes threshold: 126 mg/dL', 126),
    'LBXTC':   ('Total Cholesterol (mg/dL)', 'High threshold: 200 mg/dL', 200),
    'LBDHDD':  ('HDL Cholesterol (mg/dL)', 'Low threshold: 40 mg/dL', 40),
    'LBXHSCRP':('hsCRP (mg/L)', 'High inflammation: 3 mg/L', 3),
    'LBXVIDMS':('Vitamin D (nmol/L)', 'Deficiency: <50 nmol/L', 50),
}
lab_plot_cols = {k: v for k, v in lab_plot_cols.items() if k in nhanes_clean.columns}

if lab_plot_cols:
    n_plots = len(lab_plot_cols)
    cols    = min(3, n_plots)
    rows    = (n_plots + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
    axes = np.array(axes).flatten()

    for i, (col, (label, thresh_label, thresh)) in enumerate(lab_plot_cols.items()):
        data = nhanes_clean[col].dropna()
        cap  = data.quantile(0.97)
        data_c = data[data <= cap]
        axes[i].hist(data_c, bins=40, color=PAL[i % len(PAL)], alpha=0.8, edgecolor='white', linewidth=0.3)
        axes[i].axvline(thresh, color=CORAL, linestyle='--', linewidth=1.5, label=thresh_label)
        axes[i].axvline(data.median(), color='black', linestyle=':', linewidth=1, alpha=0.6,
                         label=f'Median: {data.median():.1f}')
        axes[i].set_title(label, fontsize=10)
        axes[i].legend(fontsize=7)

    for j in range(len(lab_plot_cols), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Lab Biomarker Distributions — NHANES 2021–2023', fontsize=14, fontweight='bold')
    plt.tight_layout()
    save('14_lab_biomarkers')
else:
    print('Lab columns not available after merging — skipping this plot.')

---
## 12. Outlier Detection

In [ ]:
def detect_outliers(df, col):
    data = df[col].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out = ((data < lo) | (data > hi)).sum()
    return n_out, lo, hi, data.min(), data.max()

print('Outlier report (IQR method):')
print(f'{'Column':<20} {'Outliers':>10} {'Lower':>10} {'Upper':>10} {'Min':>10} {'Max':>10}')
print('-'*72)

outlier_check_cols = ['BMXBMI','DR1TKCAL','DR1TPROT','DR1TCARB','DR1TTFAT','DR1TFIBE']
outlier_check_cols = [c for c in outlier_check_cols if c in nhanes_clean.columns]

for col in outlier_check_cols:
    n, lo, hi, mn, mx = detect_outliers(nhanes_clean, col)
    pct = n / nhanes_clean[col].notna().sum() * 100
    flag = ' ⚠' if pct > 5 else ''
    print(f'{col:<20} {n:>7} ({pct:.1f}%) {lo:>10.1f} {hi:>10.1f} {mn:>10.1f} {mx:>10.1f}{flag}')

print('\nFood DB outliers:')
for col in ['Caloric Value','Protein','Fat','Carbohydrates']:
    if col in food_raw.columns:
        n, lo, hi, mn, mx = detect_outliers(food_raw, col)
        pct = n / food_raw[col].notna().sum() * 100
        print(f'{col:<25} {n:>7} ({pct:.1f}%)')

---
## 13. Feature Readiness Summary (for ML training)

In [ ]:
ml_features = {
    'BMXBMI':        'Input feature — user BMI',
    'has_diabetes':  'Input feature — disease flag',
    'has_hypertension': 'Input feature — disease flag',
    'has_malnutrition':  'Input feature — derived from BMI',
    'DR1TKCAL':      'Target — daily calories',
    'DR1TPROT':      'Target — protein g/day',
    'DR1TCARB':      'Target — carbs g/day',
    'DR1TTFAT':      'Target — fat g/day',
    'DR1TFIBE':      'Target — fiber g/day',
}

print('ML Feature Readiness Report')
print('='*65)
print(f'{'Column':<22} {'Role':<35} {'Available':>9} {'Missing%':>8}')
print('-'*65)

for col, role in ml_features.items():
    if col in nhanes_clean.columns:
        miss_pct = nhanes_clean[col].isna().mean() * 100
        avail    = nhanes_clean[col].notna().sum()
        status   = '✓' if miss_pct < 20 else '⚠  HIGH' if miss_pct < 50 else '✗  DROP'
        print(f'{col:<22} {role:<35} {avail:>9,} {miss_pct:>7.1f}%  {status}')
    else:
        print(f'{col:<22} {role:<35} {"NOT FOUND":>9}')

print()
clean_rows = nhanes_clean.dropna(subset=[c for c in ['BMXBMI','DR1TKCAL','DR1TPROT','DR1TCARB','DR1TTFAT'] if c in nhanes_clean.columns])
print(f'Rows with all ML columns present: {len(clean_rows):,} / {len(nhanes_clean):,} ({100*len(clean_rows)/len(nhanes_clean):.1f}%)')
print(f'\n→ This is your usable training set size for Group A train.py')

---
## 14. Export Summary & Save Final Merged CSVs to Drive

In [ ]:
# Save cleaned NHANES dataset
nhanes_out_path = os.path.join(OUT_FOLDER, 'nhanes_clean_eda.csv')
nhanes_clean.to_csv(nhanes_out_path, index=False)
print(f'Saved nhanes_clean_eda.csv  ({len(nhanes_clean):,} rows)')

# Save merged food groups
food_out_path = os.path.join(OUT_FOLDER, 'food_groups_merged.csv')
food_raw.to_csv(food_out_path, index=False)
print(f'Saved food_groups_merged.csv ({len(food_raw):,} rows)')

# EDA text summary
summary_lines = [
    '=== EDA SUMMARY REPORT ===',
    f'Generated for: COMP6056001 AI Meal Planner — Group A',
    '',
    '--- DATASET SHAPES ---',
    f'NHANES 2021-23 (merged):   {nhanes.shape[0]:,} rows × {nhanes.shape[1]} cols',
    f'NHANES (after cleaning):   {nhanes_clean.shape[0]:,} rows',
    f'FOOD-DATA-GROUP (merged):  {food_raw.shape[0]:,} rows × {food_raw.shape[1]} cols',
    f'Daily food nutrition:      {meal_df.shape[0]:,} rows × {meal_df.shape[1]} cols',
    f'Combined laboratory:       {lab_df.shape[0]:,} rows × {lab_df.shape[1]} cols',
    '',
    '--- BMI STATS ---',
    f'Mean BMI:    {nhanes_clean["BMXBMI"].mean():.2f}' if 'BMXBMI' in nhanes_clean.columns else '',
    f'Median BMI:  {nhanes_clean["BMXBMI"].median():.2f}' if 'BMXBMI' in nhanes_clean.columns else '',
    f'Std BMI:     {nhanes_clean["BMXBMI"].std():.2f}'   if 'BMXBMI' in nhanes_clean.columns else '',
    '',
    '--- DISEASE PREVALENCE ---',
]
for col, label in [('has_diabetes','Diabetes'),('has_hypertension','Hypertension'),('has_malnutrition','Malnutrition')]:
    if col in nhanes_clean.columns:
        pct = nhanes_clean[col].mean()*100
        summary_lines.append(f'{label}: {pct:.1f}%')

summary_lines += [
    '',
    '--- PLOTS SAVED ---',
    '01_nhanes_missing_values.png',
    '02_food_missing_values.png',
    '03_nhanes_missing_matrix.png',
    '04_bmi_distribution.png',
    '05_disease_prevalence.png',
    '06_bmi_vs_disease.png',
    '07_macro_distributions.png',
    '08_macros_by_bmi_category.png',
    '09_correlation_heatmap.png',
    '10_food_nutrient_distributions.png',
    '11_food_caloric_extremes.png',
    '12_food_protein_vs_calories.png',
    '13_meal_type_analysis.png',
    '14_lab_biomarkers.png',
]

summary_path = os.path.join(OUT_FOLDER, 'EDA_Summary.txt')
with open(summary_path, 'w') as f:
    f.write('\n'.join(summary_lines))
print(f'Saved EDA_Summary.txt')

print(f'\n=== ALL FILES SAVED TO: {OUT_FOLDER} ===')
print('Files in EDA_Results:')
for f in sorted(os.listdir(OUT_FOLDER)):
    size = os.path.getsize(os.path.join(OUT_FOLDER, f)) / 1024
    print(f'  {f:<40} {size:.0f} KB')